In [51]:
import numpy as np
import xarray as xr
import pandas as pd
import os.path

# Baseline Model

## Table of Contents
- [Model Choice](#1-model-choice)
- [Data & Model Execution](#2-data--model-execution)
- [Evaluation](#3-evaluation)


## **1.** Model Choice

The Persistence Model has be chosen as the baseline mdoel. In meteorology, this is the most fundamental benchmark.</br>
It assumes that the atmospheric state remains constant over time ($ŷ_{t+1} = x_t$).</br>
For high-frequency data (hourly), persistence is a very strong baseline due to high temporal autocorrelation.</br>
Any ML model must outperform this 'no-change' assumption to prove its utility.


## **2.** Data & Model Execution

This analysis focuses on two target variables: zonal wind (u) and meridional wind (v).</br>
The implementation involves shifting the time series by one hour ($lag=1$) to create a prediction.</br>
In accordance with the experiments, the data is normalized using Z-score normalization (mean=0, std=1) to ensure the baseline MSE is directly comparable to the loss values produced by our Transformer model during training.


In [52]:
path = '../1_DatasetCharacteristics/data'

In [53]:
file = 'GFS_full_dataset.nc'
gfs_data = xr.open_dataset(os.path.join(path, file))

In [54]:
# Convert the Xarray into a DataFrame.
variables = ['zonal_wind', 'meridional_wind']
y = gfs_data[variables].to_dataframe()[variables].dropna()

y_normalized = (y - y.mean()) / y.std()
# y_true includes all values from t=1 to N
# y_pred includes all values t=0 ti N-1 (acting as the predicted values for the time steps from t=1 to N)
y_true = y_normalized.iloc[1:].values
y_pred = y_normalized.iloc[:-1].values

# 3. Calculate the MSE for the baseline model via MSE = (1 / N) * Σ (yᵢ - ŷᵢ)².
baseline_mse = np.power(y_true - y_pred, 2).mean(axis=0)

## **3.** Evaluation

The primary evaluation metric is the Mean Squared Error (MSE) tocaculate the average of the squared differences between the actual observed value and the persistence prediction.</br>
Since the MSE is the standard loss function for regression tasks and is also used in our Transformer training (via Gaussian NLL or direct MSE), it allows for a transparent performance comparison.


In [55]:
for idx, var in enumerate(variables):
    print(f'MSE = {baseline_mse[idx]:.6f} for `{var}`')

total_mse = baseline_mse.mean()
print("-" * 36)
print(f'Total Average Baseline MSE: {total_mse:.6f}')

MSE = 0.012778 for `zonal_wind`
MSE = 0.022014 for `meridional_wind`
------------------------------------
Total Average Baseline MSE: 0.017396
